In [8]:
pip install gradio scikit-learn numpy matplotlib

In [10]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # GUI 없는 환경(서버)에서 그래프 생성을 위해 필수
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

# 한글 폰트 설정 (시스템에 한글 폰트가 없을 경우 그래프에서 깨질 수 있음)
# 필요 시 matplotlib.rc('font', family='NanumGothic') 등을 추가하세요.

# 1. 모델 학습 (앱 시작 시 1회만 실행)
print("📦 데이터 로딩 및 모델 학습 중...")

CATEGORIES = ["comp.graphics", "sci.space", "talk.religion.misc"]
CAT_LABELS = ["🖥️ 컴퓨터 그래픽", "🚀 우주·과학", "✝️ 종교·철학"]
CAT_COLORS = ["#378ADD", "#1D9E75", "#D85A30"]
CAT_KEYWORDS = {
    "comp.graphics": ["image","graphics","jpeg","color","software","display",
                      "format","pixel","rendering","opengl","texture","3d"],
    "sci.space": ["space","nasa","orbit","shuttle","earth","moon",
                  "satellite","mission","launch","rocket","astronaut","mars"],
    "talk.religion.misc": ["god","christian","church","jesus","bible","religion",
                            "faith","moral","prayer","sin","spiritual","atheist"],
}

# 데이터 불러오기
raw = fetch_20newsgroups(
    subset="all",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=42,
)

train_data, train_labels = [], []
for i, cat in enumerate(CATEGORIES):
    # 각 카테고리별로 상위 100개씩만 학습 (속도 최적화)
    idxs = np.where(raw.target == i)[0][:100]
    for idx in idxs:
        train_data.append(raw.data[idx])
        train_labels.append(cat)

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=30000)
train_matrix = vectorizer.fit_transform(train_data)
vocab = set(vectorizer.get_feature_names_out())

print(f"✅ 학습 완료! 문서 {len(train_data)}개 | 어휘 {len(vocab):,}개\n")

# 2. 분류 로직
def classify(text: str):
    if not text.strip():
        return None, {cat: 0.0 for cat in CATEGORIES}, []

    input_vec = vectorizer.transform([text])
    sims = cosine_similarity(input_vec, train_matrix)[0]

    cat_scores = {}
    for i, cat in enumerate(CATEGORIES):
        mask = np.array([l == cat for l in train_labels])
        # 해당 카테고리 문서들과의 평균 유사도 계산
        cat_scores[cat] = float(np.mean(sims[mask]))

    tokens = text.lower().replace(".", " ").replace(",", " ").split()
    oov = [t for t in tokens if t and t not in ENGLISH_STOP_WORDS and t not in vocab]

    predicted = max(cat_scores, key=cat_scores.get)
    return predicted, cat_scores, oov

# 3. 시각화/HTML 생성 함수 (기존 코드와 동일)
def make_bar_chart(cat_scores: dict):
    fig, ax = plt.subplots(figsize=(6, 2.8))
    scores = [cat_scores[cat] for cat in CATEGORIES]
    max_idx = scores.index(max(scores))

    bars = ax.barh(CAT_LABELS, scores, color=[CAT_COLORS[i] if i == max_idx else f"{CAT_COLORS[i]}66" for i in range(3)])
    ax.spines[["top","right","left"]].set_visible(False)
    plt.tight_layout()
    return fig

def make_oov_html(text: str, oov_words: list) -> str:
    if not text.strip(): return ""
    oov_set = set(oov_words)
    parts = []
    for t in text.split():
        clean = t.lower().strip(".,!?;:\"'()")
        if clean in oov_set:
            parts.append(f'<span style="background:#FAEEDA; color:#633806; padding:2px 4px; border-radius:4px;">{t} ⚠</span>')
        elif clean in ENGLISH_STOP_WORDS:
            parts.append(f'<span style="color:#aaaaaa;">{t}</span>')
        else:
            parts.append(f'<span style="background:#EAF3DE; color:#27500A; padding:2px 4px; border-radius:4px;">{t}</span>')
    return f'<div style="line-height:2; font-size:14px;">{" ".join(parts)}</div>'

def make_keyword_html(predicted: str) -> str:
    if not predicted: return ""
    idx = CATEGORIES.index(predicted)
    color = CAT_COLORS[idx]
    badges = "".join([f'<span style="background:{color}22; color:{color}; border:1px solid {color}55; padding:2px 8px; border-radius:12px; margin-right:5px;">{k}</span>' for k in CAT_KEYWORDS[predicted]])
    return f'<div style="margin-top:10px;">{badges}</div>'

def run_classify(text: str):
    predicted, cat_scores, oov = classify(text)
    top_score = max(cat_scores.values())

    if top_score == 0:
        return "❌ 분류 불가", 0, make_bar_chart(cat_scores), make_oov_html(text, oov), ""

    res_label = f"{CAT_LABELS[CATEGORIES.index(predicted)]} ({top_score:.4f})"
    return res_label, round(top_score*100, 2), make_bar_chart(cat_scores), make_oov_html(text, oov), make_keyword_html(predicted)

# 4. Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📰 뉴스그룹 텍스트 분류기")
    with gr.Row():
        with gr.Column():
            txt_input = gr.Textbox(label="영어 텍스트 입력", lines=5)
            btn = gr.Button("분류하기", variant="primary")
        with gr.Column():
            out_label = gr.Textbox(label="예측 결과")
            out_score = gr.Number(label="신뢰도 (%)")
            out_chart = gr.Plot(label="유사도 분석")
    with gr.Row():
        out_oov = gr.HTML(label="단어 분석")
        out_kw = gr.HTML(label="키워드")

    btn.click(run_classify, inputs=txt_input, outputs=[out_label, out_score, out_chart, out_oov, out_kw])

if __name__ == "__main__":
    demo.launch(share=True)

📦 데이터 로딩 및 모델 학습 중...
✅ 학습 완료! 문서 300개 | 어휘 30,000개



/tmp/ipykernel_551/749017462.py:114: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e066432b1fa4822de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
"""
배트맨 vs 조커: kill/save 단어로 CountVectorize → 코사인 유사도
==============================================================
과제 핵심:
  X축 = 'save' 출현 횟수
  Y축 = 'kill' 출현 횟수
  각 대사(문장) 한 줄이 하나의 데이터 포인트
  캐릭터별 집계 벡터 간 코사인 유사도 계산
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ────────────────────────────────────────────
# 1. 가상 대사 데이터
#    배트맨: save 위주 / 조커: kill 위주
# ────────────────────────────────────────────
batman_lines = [
    "I will save this city from crime",
    "save the innocent people save Gotham",
    "we must save them before it is too late",
    "save every life that is what I do",
    "I will save you I promise to save everyone",
    "save Gotham save the people never kill",
    "I do not kill I save",
    "save the hostages now save them all",
    "my mission is to save not to kill",
    "save Gotham save Alfred save everyone I love",
    "kill is not an option I save lives",
    "save the day save the night save always",
]

joker_lines = [
    "kill kill kill that is all that matters",
    "why save anyone kill them all hahaha",
    "I will kill everyone in this city",
    "kill the batman kill the cops kill kill",
    "to kill is to be free kill freely",
    "save nothing kill everything burn it down",
    "kill for fun kill for chaos always kill",
    "I only kill I never save anyone",
    "kill the order kill the system kill",
    "why do heroes save just kill instead",
    "kill kill chaos kill madness kill again",
    "to save is weakness to kill is power",
]

all_lines  = batman_lines + joker_lines
characters = ["Batman"] * len(batman_lines) + ["Joker"] * len(joker_lines)

# ────────────────────────────────────────────
# 2. CountVectorizer: 어휘를 kill/save 두 단어만으로 제한
# ────────────────────────────────────────────
vectorizer = CountVectorizer(vocabulary=["save", "kill"])   # X축=save, Y축=kill
count_matrix = vectorizer.fit_transform(all_lines).toarray()

print("어휘 사전:", vectorizer.vocabulary_)   # {'save': 0, 'kill': 1}
print(f"행렬 크기: {count_matrix.shape}  (문장 수 × 2)")
print()

# 각 문장의 (save 횟수, kill 횟수) 출력
print(f"{'캐릭터':<10} {'대사(앞 40자)':<42} {'save':>5} {'kill':>5}")
print("-" * 65)
for i, (line, char) in enumerate(zip(all_lines, characters)):
    s, k = count_matrix[i]
    print(f"{char:<10} {line[:40]:<42} {s:>5} {k:>5}")

# ────────────────────────────────────────────
# 3. 캐릭터별 집계 벡터 (합산)
# ────────────────────────────────────────────
n_batman = len(batman_lines)
batman_vec = count_matrix[:n_batman].sum(axis=0)   # [save합, kill합]
joker_vec  = count_matrix[n_batman:].sum(axis=0)

print(f"\n배트맨 집계 벡터 → save: {batman_vec[0]}, kill: {batman_vec[1]}")
print(f"조커   집계 벡터 → save: {joker_vec[0]},  kill: {joker_vec[1]}")

# ────────────────────────────────────────────
# 4. 코사인 유사도 계산
# ────────────────────────────────────────────
sim = cosine_similarity([batman_vec], [joker_vec])[0][0]

# 수식으로도 검증
dot   = float(np.dot(batman_vec, joker_vec))
norm  = float(np.linalg.norm(batman_vec) * np.linalg.norm(joker_vec))
sim_manual = dot / norm

print(f"\n코사인 유사도 (sklearn):  {sim:.4f}")
print(f"코사인 유사도 (수동계산): {sim_manual:.4f}")
print(f"내적(A·B):              {dot:.1f}")
print(f"||A||×||B||:            {norm:.2f}")
print(f"각도(θ):                {np.degrees(np.arccos(np.clip(sim,-1,1))):.1f}°")

# ────────────────────────────────────────────
# 5. 시각화
# ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Batman vs Joker: kill/save CountVectorizer + Cosine Similarity",
             fontsize=14, fontweight="bold", y=1.01)

# ── 왼쪽: 산점도 (각 대사가 하나의 점)
ax = axes[0]
bat_pts = count_matrix[:n_batman]
jok_pts = count_matrix[n_batman:]

# 겹치는 점은 크기로 구분 (jitter 추가)
np.random.seed(42)
jitter = 0.12

ax.scatter(bat_pts[:, 0] + np.random.uniform(-jitter, jitter, len(bat_pts)),
           bat_pts[:, 1] + np.random.uniform(-jitter, jitter, len(bat_pts)),
           c="#1565C0", s=80, alpha=0.75, label="Batman", edgecolors="white", linewidths=0.5, zorder=3)

ax.scatter(jok_pts[:, 0] + np.random.uniform(-jitter, jitter, len(jok_pts)),
           jok_pts[:, 1] + np.random.uniform(-jitter, jitter, len(jok_pts)),
           c="#B71C1C", s=80, alpha=0.75, label="Joker", edgecolors="white", linewidths=0.5, zorder=3)

# 집계 벡터 화살표
ax.annotate("", xy=(batman_vec[0], batman_vec[1]), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="#1565C0", lw=2.5))
ax.annotate("", xy=(joker_vec[0], joker_vec[1]), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="#B71C1C", lw=2.5))
ax.text(batman_vec[0]+0.3, batman_vec[1]+0.3, f"Batman\n({batman_vec[0]}, {batman_vec[1]})",
        color="#1565C0", fontweight="bold", fontsize=9)
ax.text(joker_vec[0]+0.3,  joker_vec[1]-1.5,  f"Joker\n({joker_vec[0]}, {joker_vec[1]})",
        color="#B71C1C", fontweight="bold", fontsize=9)

ax.set_xlabel("save 출현 횟수", fontsize=11)
ax.set_ylabel("kill 출현 횟수", fontsize=11)
ax.set_title("각 대사 = 하나의 데이터 포인트", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
ax.set_xlim(-0.5)
ax.set_ylim(-0.5)
ax.spines[["top","right"]].set_visible(False)

# ── 오른쪽: 정규화 벡터 + 각도 표시
ax2 = axes[1]
b_norm = batman_vec / np.linalg.norm(batman_vec)
j_norm = joker_vec  / np.linalg.norm(joker_vec)

ax2.annotate("", xy=(b_norm[0], b_norm[1]), xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color="#1565C0", lw=2.5))
ax2.annotate("", xy=(j_norm[0], j_norm[1]), xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color="#B71C1C", lw=2.5))

# 각도 호 그리기
theta1 = np.degrees(np.arctan2(b_norm[1], b_norm[0]))
theta2 = np.degrees(np.arctan2(j_norm[1], j_norm[0]))
arc = mpatches.Arc((0, 0), 0.4, 0.4, angle=0, theta1=min(theta1, theta2),
                   theta2=max(theta1, theta2), color="#555555", lw=1.5)
ax2.add_patch(arc)
mid_theta = np.radians((theta1 + theta2) / 2)
ax2.text(0.25 * np.cos(mid_theta), 0.25 * np.sin(mid_theta),
         f"θ={np.degrees(np.arccos(np.clip(sim,-1,1))):.1f}°",
         ha="center", va="center", fontsize=10, color="#333333")

ax2.text(b_norm[0]+0.03, b_norm[1]+0.03, "Batman (정규화)", color="#1565C0",
         fontweight="bold", fontsize=9)
ax2.text(j_norm[0]+0.03, j_norm[1]-0.06, "Joker (정규화)", color="#B71C1C",
         fontweight="bold", fontsize=9)

ax2.set_xlim(-0.1, 1.2)
ax2.set_ylim(-0.1, 1.2)
ax2.set_xlabel("save 방향 (정규화)", fontsize=11)
ax2.set_ylabel("kill 방향 (정규화)", fontsize=11)
ax2.set_title(f"코사인 유사도 = cos(θ) = {sim:.4f}", fontsize=12, fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("batman_joker_cosine.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n그래프 저장 완료: batman_joker_cosine.png")

# ────────────────────────────────────────────
# 6. 해설 요약
# ────────────────────────────────────────────
print(f"""
{'='*55}
[해설] 왜 코사인 유사도는 {sim:.4f}인가?
{'='*55}
배트맨 벡터: save={batman_vec[0]}, kill={batman_vec[1]}
  → save를 많이 쓰므로 X축(save) 방향에 가깝게 위치

조커 벡터:   save={joker_vec[0]}, kill={joker_vec[1]}
  → kill을 많이 쓰므로 Y축(kill) 방향에 가깝게 위치

두 벡터의 방향이 서로 다르므로 → θ = {np.degrees(np.arccos(np.clip(sim,-1,1))):.1f}°
코사인 유사도 = cos({np.degrees(np.arccos(np.clip(sim,-1,1))):.1f}°) ≈ {sim:.4f}

※ 유사도가 0에 가까울수록 두 캐릭터의 언어 패턴이 다름을 의미합니다.
""")

어휘 사전: {'save': 0, 'kill': 1}
행렬 크기: (24, 2)  (문장 수 × 2)

캐릭터        대사(앞 40자)                                   save  kill
-----------------------------------------------------------------
Batman     I will save this city from crime               1     0
Batman     save the innocent people save Gotham           2     0
Batman     we must save them before it is too late        1     0
Batman     save every life that is what I do              1     0
Batman     I will save you I promise to save everyo       2     0
Batman     save Gotham save the people never kill         2     1
Batman     I do not kill I save                           1     1
Batman     save the hostages now save them all            2     0
Batman     my mission is to save not to kill              1     1
Batman     save Gotham save Alfred save everyone I        3     0
Batman     kill is not an option I save lives             1     1
Batman     save the day save the night save always        3     0
Joker      kill ki

/tmp/ipykernel_551/2707719827.py:176: UserWarning: Glyph 52636 (\N{HANGUL SYLLABLE CUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_551/2707719827.py:176: UserWarning: Glyph 54788 (\N{HANGUL SYLLABLE HYEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_551/2707719827.py:176: UserWarning: Glyph 54943 (\N{HANGUL SYLLABLE HOES}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_551/2707719827.py:176: UserWarning: Glyph 49688 (\N{HANGUL SYLLABLE SU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_551/2707719827.py:176: UserWarning: Glyph 44033 (\N{HANGUL SYLLABLE GAG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_551/2707719827.py:176: UserWarning: Glyph 45824 (\N{HANGUL SYLLABLE DAE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_551/2707719827.py:176: UserWarning: Glyph 49324 (\N{HANGUL SYLLABLE SA}) missing from font(s) DejaVu Sans.
  plt.tight_


그래프 저장 완료: batman_joker_cosine.png

[해설] 왜 코사인 유사도는 0.3846인가?
배트맨 벡터: save=20, kill=4
  → save를 많이 쓰므로 X축(save) 방향에 가깝게 위치

조커 벡터:   save=5, kill=25
  → kill을 많이 쓰므로 Y축(kill) 방향에 가깝게 위치

두 벡터의 방향이 서로 다르므로 → θ = 67.4°
코사인 유사도 = cos(67.4°) ≈ 0.3846

※ 유사도가 0에 가까울수록 두 캐릭터의 언어 패턴이 다름을 의미합니다.

